In [1]:
import requests
r = requests.get("http://www.khs.go.kr/cha/SearchKindOpenapiList.do", params={
    "pageUnit": 1,
    "pageIndex": 1,
    "ccbaCncl": "N",
    "ccbaCtcd": "11"
})
print(r.text)

<?xml version="1.0" encoding="utf-8"?>







<result>
<totalCnt>2115</totalCnt>
<pageUnit>1</pageUnit>
<pageIndex>1</pageIndex>	

<item>
<sn>1</sn>
<no>1</no>
<ccmaName><![CDATA[국보]]></ccmaName>
<ccbaMnm1><![CDATA[서울 숭례문]]></ccbaMnm1>
<ccbaMnm2><![CDATA[서울 崇禮門]]></ccbaMnm2>
<ccbaCtcdNm><![CDATA[서울]]></ccbaCtcdNm>
<ccsiName><![CDATA[중구]]></ccsiName>
<ccbaAdmin><![CDATA[국가유산청 덕수궁관리소]]></ccbaAdmin>
<ccbaKdcd>11</ccbaKdcd>
<ccbaCtcd>11</ccbaCtcd>
<ccbaAsno>0000010000000</ccbaAsno>
<ccbaCncl>N</ccbaCncl>
<ccbaCpno>1111100010000</ccbaCpno>
<longitude>126.975312652739</longitude>
<latitude>37.559975221378</latitude>
<regDt>2025-06-26 18:46:08</regDt>

</item>
	
</result>



# 데이터 수집

# # 바로 실행
df = main()

# 또는 개별 실행
heritage_df = collect_national_heritage_all()  # 국가유산만
gung_df = collect_gung_all()  # 궁궐·종묘만

In [5]:
import requests
import pandas as pd
from tqdm import tqdm
import time
import json
import logging
import xml.etree.ElementTree as ET
from urllib.parse import urlencode

# 로깅 설정
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# === 시도 코드 목록 (전국 대상) === #
SIDO_CODES = [
    "11", "21", "22", "23", "24", "25", "26", "45",
    "31", "32", "33", "34", "35", "36", "37", "38", "50"
]

# 시도 코드명 매핑
SIDO_NAMES = {
    "11": "서울", "21": "부산", "22": "대구", "23": "인천", 
    "24": "광주", "25": "대전", "26": "울산", "45": "세종",
    "31": "경기", "32": "강원", "33": "충북", "34": "충남", 
    "35": "전북", "36": "전남", "37": "경북", "38": "경남", "50": "제주"
}

def safe_api_call(func, *args, **kwargs):
    """API 호출을 안전하게 처리하는 래퍼 함수"""
    max_retries = 3
    for attempt in range(max_retries):
        try:
            return func(*args, **kwargs)
        except requests.exceptions.RequestException as e:
            logger.warning(f"API 호출 실패 (시도 {attempt + 1}/{max_retries}): {e}")
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)  # 지수 백오프
            else:
                logger.error(f"최대 재시도 횟수 초과: {e}")
                return None
        except Exception as e:
            logger.error(f"예상치 못한 오류: {e}")
            return None

def parse_xml_response(xml_text):
    """XML 응답을 파싱하여 딕셔너리로 변환"""
    try:
        root = ET.fromstring(xml_text)
        
        # 공통적인 XML 구조 처리
        result = {"data": []}
        
        # item 태그들을 찾아서 처리
        items = root.findall('.//item')
        if not items:
            # 다른 구조일 수 있으므로 모든 하위 요소 확인
            items = root.findall('.//*')
            if len(items) <= 3:  # 루트 요소만 있는 경우
                return {"data": []}
        
        data_list = []
        for item in items:
            if item.tag == 'item' or len(list(item)) > 0:  # item 태그이거나 하위 요소가 있는 경우
                item_dict = {}
                for child in item:
                    item_dict[child.tag] = child.text if child.text else ""
                if item_dict:  # 빈 딕셔너리가 아닌 경우만 추가
                    data_list.append(item_dict)
        
        result["data"] = data_list
        return result
        
    except ET.ParseError as e:
        logger.error(f"XML 파싱 오류: {e}")
        return {"data": []}
    except Exception as e:
        logger.error(f"XML 처리 중 오류: {e}")
        return {"data": []}

def test_api_response(url, params):
    """API 응답 형식을 테스트하는 함수"""
    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        
        content_type = response.headers.get('content-type', '')
        logger.info(f"API 응답 테스트:")
        logger.info(f"Status Code: {response.status_code}")
        logger.info(f"Content-Type: {content_type}")
        logger.info(f"Response length: {len(response.text)}")
        logger.info(f"First 200 chars: {response.text[:200]}")
        
        return response.text, content_type
        
    except Exception as e:
        logger.error(f"API 테스트 실패: {e}")
        return None, None

# === 국가유산 목록 API === #
def fetch_heritage_list(page=1, per_page=50, ccbaCtcd='11'):
    """국가유산 목록을 가져오는 함수"""
    url = "http://www.khs.go.kr/cha/SearchKindOpenapiList.do"
    params = {
        "pageUnit": per_page,
        "pageIndex": page,
        "ccbaCncl": "N",  # 해제되지 않은 유산만
        "ccbaCtcd": ccbaCtcd
    }
    
    logger.info(f"국가유산 목록 요청: 시도코드={ccbaCtcd}, 페이지={page}")
    
    # 첫 페이지인 경우 API 응답 형식 테스트
    if page == 1:
        test_response, test_content_type = test_api_response(url, params)
        if test_response is None:
            return {"data": []}
    
    try:
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        
        # 응답 내용 확인
        if not response.text.strip():
            logger.warning("빈 응답을 받았습니다.")
            return {"data": []}
        
        # Content-Type 확인 및 파싱
        content_type = response.headers.get('content-type', '')
        
        if 'application/xml' in content_type or 'text/xml' in content_type:
            logger.info("XML 응답을 파싱합니다.")
            return parse_xml_response(response.text)
        elif 'application/json' in content_type:
            try:
                data = response.json()
                if isinstance(data, dict) and "result" in data:
                    return {"data": data.get("result", [])}
                elif isinstance(data, dict) and "data" in data:
                    return data
                elif isinstance(data, list):
                    return {"data": data}
                else:
                    return {"data": []}
            except json.JSONDecodeError as e:
                logger.error(f"JSON 파싱 오류: {e}")
                return {"data": []}
        else:
            # Content-Type이 명확하지 않은 경우, 내용을 보고 판단
            text = response.text.strip()
            if text.startswith('<?xml') or text.startswith('<'):
                logger.info("XML로 추정되는 응답을 파싱합니다.")
                return parse_xml_response(text)
            elif text.startswith('{') or text.startswith('['):
                try:
                    data = json.loads(text)
                    return {"data": data if isinstance(data, list) else [data]}
                except:
                    logger.warning("JSON 파싱 실패, 빈 결과 반환")
                    return {"data": []}
            else:
                logger.warning(f"알 수 없는 응답 형식: {content_type}")
                logger.debug(f"응답 내용: {text[:200]}")
                return {"data": []}
            
    except requests.exceptions.Timeout:
        logger.error("요청 시간 초과")
        return {"data": []}
    except requests.exceptions.RequestException as e:
        logger.error(f"요청 오류: {e}")
        return {"data": []}

# === 국가유산 상세 API === #
def fetch_heritage_detail(ccbaKdcd, ccbaAsno, ccbaCtcd):
    """국가유산 상세 정보를 가져오는 함수"""
    url = "http://www.khs.go.kr/cha/SearchKindOpenapiDt.do"
    params = {
        "ccbaKdcd": ccbaKdcd,
        "ccbaAsno": ccbaAsno,
        "ccbaCtcd": ccbaCtcd
    }
    
    try:
        response = requests.get(url, params=params, timeout=20)
        response.raise_for_status()
        
        if not response.text.strip():
            return {"data": {}}
        
        content_type = response.headers.get('content-type', '')
        
        if 'application/xml' in content_type or 'text/xml' in content_type:
            parsed = parse_xml_response(response.text)
            # 상세 정보는 보통 단일 아이템이므로 첫 번째 아이템 반환
            data_list = parsed.get("data", [])
            return {"data": data_list[0] if data_list else {}}
        elif 'application/json' in content_type:
            data = response.json()
            if isinstance(data, dict) and "result" in data:
                return {"data": data.get("result", {})}
            elif isinstance(data, dict) and "data" in data:
                return data
            else:
                return {"data": data if isinstance(data, dict) else {}}
        else:
            # 자동 감지
            text = response.text.strip()
            if text.startswith('<?xml') or text.startswith('<'):
                parsed = parse_xml_response(text)
                data_list = parsed.get("data", [])
                return {"data": data_list[0] if data_list else {}}
            else:
                return {"data": {}}
            
    except Exception as e:
        logger.warning(f"상세 정보 가져오기 실패: {e}")
        return {"data": {}}

# === 국가유산 이미지 API === #
def fetch_heritage_image(ccbaKdcd, ccbaAsno, ccbaCtcd):
    """국가유산 이미지 URL을 가져오는 함수"""
    url = "http://www.khs.go.kr/cha/SearchImageOpenapi.do"
    params = {
        "ccbaKdcd": ccbaKdcd,
        "ccbaAsno": ccbaAsno,
        "ccbaCtcd": ccbaCtcd
    }
    
    try:
        response = requests.get(url, params=params, timeout=15)
        response.raise_for_status()
        
        content_type = response.headers.get('content-type', '')
        
        if 'application/xml' in content_type or 'text/xml' in content_type:
            parsed = parse_xml_response(response.text)
            result = parsed.get("data", [])
        elif 'application/json' in content_type:
            data = response.json()
            if isinstance(data, dict) and "data" in data:
                result = data.get("data", [])
            elif isinstance(data, list):
                result = data
            else:
                result = []
        else:
            text = response.text.strip()
            if text.startswith('<?xml') or text.startswith('<'):
                parsed = parse_xml_response(text)
                result = parsed.get("data", [])
            else:
                result = []
        
        return result[0].get("imageUrl", "") if result else ""
        
    except Exception:
        return ""

# === 국가유산 동영상 API === #
def fetch_heritage_video(ccbaKdcd, ccbaAsno, ccbaCtcd):
    """국가유산 동영상 URL을 가져오는 함수"""
    url = "http://www.khs.go.kr/cha/SearchVideoOpenapi.do"
    params = {
        "ccbaKdcd": ccbaKdcd,
        "ccbaAsno": ccbaAsno,
        "ccbaCtcd": ccbaCtcd
    }
    
    try:
        response = requests.get(url, params=params, timeout=15)
        response.raise_for_status()
        
        content_type = response.headers.get('content-type', '')
        
        if 'application/xml' in content_type or 'text/xml' in content_type:
            parsed = parse_xml_response(response.text)
            result = parsed.get("data", [])
        elif 'application/json' in content_type:
            data = response.json()
            if isinstance(data, dict) and "data" in data:
                result = data.get("data", [])
            elif isinstance(data, list):
                result = data
            else:
                result = []
        else:
            text = response.text.strip()
            if text.startswith('<?xml') or text.startswith('<'):
                parsed = parse_xml_response(text)
                result = parsed.get("data", [])
            else:
                result = []
            
        return result[0].get("videoUrl", "") if result else ""
        
    except Exception:
        return ""

# === 전국 국가유산 전체 수집 === #
def collect_national_heritage_all():
    """전국의 모든 국가유산 정보를 수집"""
    result = []
    total_collected = 0
    
    for ctcd in tqdm(SIDO_CODES, desc="전국 시도코드 순회"):
        sido_name = SIDO_NAMES.get(ctcd, ctcd)
        logger.info(f"{sido_name}({ctcd}) 지역 수집 시작")
        
        page = 1
        sido_count = 0
        
        while True:
            # API 호출
            data = safe_api_call(fetch_heritage_list, page=page, ccbaCtcd=ctcd)
            
            if data is None:
                logger.warning(f"{sido_name} {page}페이지 건너뜀")
                break
                
            items = data.get("data", [])
            if not items:
                logger.info(f"{sido_name} {page}페이지에서 더 이상 데이터 없음")
                break
            
            logger.info(f"{sido_name} {page}페이지: {len(items)}건 처리 중")
            
            for item in items:
                try:
                    # 상세 정보 가져오기
                    detail_data = safe_api_call(
                        fetch_heritage_detail, 
                        item.get("ccbaKdcd", ""), 
                        item.get("ccbaAsno", ""), 
                        item.get("ccbaCtcd", "")
                    )
                    
                    detail = detail_data.get("data", {}) if detail_data else {}
                    
                    # 이미지 및 동영상 URL 가져오기 (선택사항)
                    image_url = safe_api_call(
                        fetch_heritage_image,
                        item.get("ccbaKdcd", ""),
                        item.get("ccbaAsno", ""),
                        item.get("ccbaCtcd", "")
                    ) or ""
                    
                    video_url = safe_api_call(
                        fetch_heritage_video,
                        item.get("ccbaKdcd", ""),
                        item.get("ccbaAsno", ""),
                        item.get("ccbaCtcd", "")
                    ) or ""
                    
                    # 데이터 정리
                    heritage_item = {
                        "title": detail.get("ccbaMnm1") or item.get("ccbaMnm1", ""),
                        "title_hanja": detail.get("ccbaMnm2") or item.get("ccbaMnm2", ""),
                        "type": detail.get("ccmaName") or item.get("ccmaName", ""),
                        "category": detail.get("gcodeName", ""),
                        "location": detail.get("ccbaLcad") or item.get("ccbaLcto", ""),
                        "sido": sido_name,
                        "description": detail.get("content", ""),
                        "designation_date": detail.get("ccbaAsdt", ""),
                        "owner": detail.get("ccbaPoss", ""),
                        "manager": detail.get("ccbaAdmin", ""),
                        "era": detail.get("ccceName", ""),
                        "quantity": detail.get("ccbaQuan", ""),
                        "longitude": item.get("longitude", "0"),
                        "latitude": item.get("latitude", "0"),
                        "image_url": image_url,
                        "video_url": video_url,
                        "heritage_code": item.get("ccbaKdcd", ""),
                        "management_number": item.get("ccbaAsno", ""),
                        "sido_code": item.get("ccbaCtcd", "")
                    }
                    
                    result.append(heritage_item)
                    sido_count += 1
                    total_collected += 1
                    
                    # 진행 상황 출력 (100건마다)
                    if total_collected % 100 == 0:
                        logger.info(f"총 {total_collected}건 수집 완료")
                    
                except Exception as e:
                    logger.warning(f"항목 처리 중 오류: {e}")
                    continue
            
            page += 1
            time.sleep(0.5)  # API 부하 방지
        
        logger.info(f"{sido_name} 완료: {sido_count}건 수집")
    
    logger.info(f"국가유산 수집 완료: 총 {total_collected}건")
    return pd.DataFrame(result)

# === 궁궐·종묘 수집 === #
def fetch_gung_list(gung_number=1):
    """궁궐·종묘 목록을 가져오는 함수"""
    url = "https://www.heritage.go.kr/heri/gungDetail/gogungListOpenApi.do"
    params = {"gung_number": gung_number}
    
    logger.info(f"궁궐 목록 요청: 궁번호={gung_number}")
    
    try:
        response = requests.get(url, params=params, timeout=20)
        response.raise_for_status()
        
        logger.info(f"궁궐 API 응답: Status={response.status_code}, Content-Type={response.headers.get('content-type')}")
        logger.info(f"응답 내용 미리보기: {response.text[:200]}")
        
        if not response.text.strip():
            logger.warning("빈 응답")
            return []
        
        content_type = response.headers.get('content-type', '')
        
        if 'application/xml' in content_type or 'text/xml' in content_type:
            parsed = parse_xml_response(response.text)
            return parsed.get("data", [])
        elif 'application/json' in content_type:
            data = response.json()
            return data.get("data", []) if isinstance(data, dict) else data
        else:
            # 자동 감지
            text = response.text.strip()
            if text.startswith('<?xml') or text.startswith('<'):
                parsed = parse_xml_response(text)
                return parsed.get("data", [])
            elif text.startswith('{') or text.startswith('['):
                try:
                    data = json.loads(text)
                    return data.get("data", []) if isinstance(data, dict) else data
                except:
                    return []
            else:
                logger.warning(f"알 수 없는 궁궐 API 응답 형식")
                return []
                
    except Exception as e:
        logger.warning(f"궁궐 목록 가져오기 실패 (궁번호: {gung_number}): {e}")
        return []

def fetch_gung_detail(serial_number, gung_number, detail_code):
    """궁궐·종묘 상세 정보를 가져오는 함수"""
    url = "https://www.heritage.go.kr/heri/gungDetail/gogungDetailOpenApi.do"
    params = {
        "serial_number": serial_number,
        "gung_number": gung_number,
        "detail_code": detail_code
    }
    
    try:
        response = requests.get(url, params=params, timeout=20)
        response.raise_for_status()
        
        content_type = response.headers.get('content-type', '')
        
        if 'application/xml' in content_type or 'text/xml' in content_type:
            parsed = parse_xml_response(response.text)
            data_list = parsed.get("data", [])
            return data_list[0] if data_list else {}
        elif 'application/json' in content_type:
            data = response.json()
            return data.get("data", {}) if isinstance(data, dict) else data
        else:
            text = response.text.strip()
            if text.startswith('<?xml') or text.startswith('<'):
                parsed = parse_xml_response(text)
                data_list = parsed.get("data", [])
                return data_list[0] if data_list else {}
            else:
                return {}
                
    except Exception as e:
        logger.warning(f"궁궐 상세 정보 가져오기 실패: {e}")
        return {}

def collect_gung_all():
    """모든 궁궐·종묘 정보를 수집"""
    result = []
    gung_names = {1: "경복궁", 2: "창덕궁", 3: "창경궁", 4: "덕수궁", 5: "종묘"}
    
    for gung_number in tqdm(range(1, 6), desc="궁궐/종묘 수집"):
        gung_name = gung_names[gung_number]
        logger.info(f"{gung_name} 수집 시작")
        
        list_data = safe_api_call(fetch_gung_list, gung_number)
        
        if not list_data:
            logger.warning(f"{gung_name} 목록을 가져올 수 없습니다.")
            continue
        
        for item in list_data:
            try:
                detail = safe_api_call(
                    fetch_gung_detail,
                    item.get("serial_number"),
                    item.get("gung_number"),
                    item.get("detail_code")
                ) or {}
                
                gung_item = {
                    "title": detail.get("contents_kor") or item.get("contents_kor", ""),
                    "title_hanja": "",
                    "type": "궁궐·종묘",
                    "category": gung_name,
                    "location": "서울특별시",
                    "sido": "서울",
                    "description": detail.get("explanation_kor") or item.get("explanation_kor", ""),
                    "designation_date": "",
                    "owner": "",
                    "manager": "",
                    "era": "",
                    "quantity": "",
                    "longitude": "0",
                    "latitude": "0",
                    "image_url": detail.get("imgUrl") or item.get("imgUrl", ""),
                    "video_url": detail.get("moving", ""),
                    "heritage_code": "",
                    "management_number": "",
                    "sido_code": "11"
                }
                
                result.append(gung_item)
                
            except Exception as e:
                logger.warning(f"{gung_name} 항목 처리 중 오류: {e}")
                continue
        
        logger.info(f"{gung_name} 완료")
        time.sleep(1)  # API 부하 방지
    
    logger.info(f"궁궐·종묘 수집 완료: 총 {len(result)}건")
    return pd.DataFrame(result)

# === 메인 실행 함수 === #
def main():
    """메인 실행 함수"""
    logger.info("국가유산 데이터 수집 시작")
    
    # 국가유산 수집
    try:
        heritage_df = collect_national_heritage_all()
        logger.info(f"국가유산 DataFrame 생성 완료: {len(heritage_df)}건")
    except Exception as e:
        logger.error(f"국가유산 수집 중 오류: {e}")
        heritage_df = pd.DataFrame()
    
    # 궁궐·종묘 수집
    try:
        gung_df = collect_gung_all()
        logger.info(f"궁궐·종묘 DataFrame 생성 완료: {len(gung_df)}건")
    except Exception as e:
        logger.error(f"궁궐·종묘 수집 중 오류: {e}")
        gung_df = pd.DataFrame()
    
    # 데이터 통합
    if not heritage_df.empty and not gung_df.empty:
        final_df = pd.concat([heritage_df, gung_df], ignore_index=True)
    elif not heritage_df.empty:
        final_df = heritage_df
    elif not gung_df.empty:
        final_df = gung_df
    else:
        logger.error("수집된 데이터가 없습니다.")
        return pd.DataFrame()
    
    # 데이터 정리
    final_df = final_df.fillna("")  # NaN 값을 빈 문자열로 변경
    final_df = final_df.drop_duplicates(subset=['title', 'location'], keep='first')  # 중복 제거
    
    # CSV 저장
    filename = "heritage_rag_full.csv"
    final_df.to_csv(filename, index=False, encoding='utf-8-sig')
    
    logger.info(f"수집 완료: 총 {len(final_df)}건")
    logger.info(f"파일 저장: {filename}")
    
    # 데이터 미리보기
    print("\n=== 수집된 데이터 미리보기 ===")
    print(f"총 데이터 수: {len(final_df)}")
    print(f"컬럼: {list(final_df.columns)}")
    if len(final_df) > 0:
        print("\n상위 5개 데이터:")
        print(final_df[['title', 'type', 'sido', 'location']].head())
    
    return final_df

# 실행
if __name__ == "__main__":
    df = main()

INFO:__main__:국가유산 데이터 수집 시작
전국 시도코드 순회:   0%|           | 0/17 [00:00<?, ?it/s]INFO:__main__:서울(11) 지역 수집 시작
INFO:__main__:국가유산 목록 요청: 시도코드=11, 페이지=1
INFO:__main__:API 응답 테스트:
INFO:__main__:Status Code: 200
INFO:__main__:Content-Type: application/xml; charset=utf-8
INFO:__main__:Response length: 28058
INFO:__main__:First 200 chars: <?xml version="1.0" encoding="utf-8"?>







<result>
<totalCnt>2115</totalCnt>
<pageUnit>50</pageUnit>
<pageIndex>1</pageIndex>	

<item>
<sn>1</sn>
<no>1</no>
<ccmaName><![CDATA[국보]]
INFO:__main__:XML 응답을 파싱합니다.
INFO:__main__:서울 1페이지: 50건 처리 중
INFO:__main__:국가유산 목록 요청: 시도코드=11, 페이지=2
INFO:__main__:XML 응답을 파싱합니다.
INFO:__main__:서울 2페이지: 50건 처리 중
INFO:__main__:총 100건 수집 완료
INFO:__main__:국가유산 목록 요청: 시도코드=11, 페이지=3
INFO:__main__:XML 응답을 파싱합니다.
INFO:__main__:서울 3페이지: 50건 처리 중
INFO:__main__:국가유산 목록 요청: 시도코드=11, 페이지=4
INFO:__main__:XML 응답을 파싱합니다.
INFO:__main__:서울 4페이지: 50건 처리 중
ERROR:__main__:XML 파싱 오류: mismatched tag: line 68, column 69
INFO:__main__:총 200건 수집 완


=== 수집된 데이터 미리보기 ===
총 데이터 수: 15515
컬럼: ['title', 'title_hanja', 'type', 'category', 'location', 'sido', 'description', 'designation_date', 'owner', 'manager', 'era', 'quantity', 'longitude', 'latitude', 'image_url', 'video_url', 'heritage_code', 'management_number', 'sido_code']

상위 5개 데이터:
               title type sido  \
0             서울 숭례문   국보   서울   
1       서울 원각사지 십층석탑   국보   서울   
2  서울 북한산 신라 진흥왕 순수비   국보   서울   
3        청자 사자형뚜껑 향로   국보   서울   
4         청자 어룡형 주전자   국보   서울   

                                            location  
0  \n\t\t\t\n\t             \n\t                 ...  
1  \n\t\t\t\n\t             \n\t                 ...  
2  \n\t\t\t\n\t             \n\t                 ...  
3  \n\t\t\t\n\t             \n\t                 ...  
4  \n\t\t\t\n\t             \n\t                 ...  


# LLM RAG 실습

In [33]:
import pandas as pd
dataset = pd.read_csv('heritage_rag_full.csv')
dataset.head(2)

,title,title_hanja,type,category,location,sido,description,designation_date,owner,manager,era,quantity,longitude,latitude,image_url,video_url,heritage_code,management_number,sido_code
0,서울 숭례문,서울 崇禮門,국보,유적건조물,\r\n\t\t\t\r\n\t \r\n\t ...,서울,조선시대 한양도성의 정문으로 남쪽에 있다고 해서 남대문이라고도 불렀다. 현재 서울에...,19621220.0,\r\n 국유\r\n,\r\n 국가유산청 덕수궁관리소\r\n\t\t,조선 태조 7년(1398),1동,126.975313,37.559975,http://www.khs.go.kr/unisearch/images/national...,http://116.67.83.213/webdata/file_data/media_d...,11.0,10000000.0,11
1,서울 원각사지 십층석탑,서울 圓覺寺址 十層石塔,국보,유적건조물,\r\n\t\t\t\r\n\t \r\n\t ...,서울,"원각사는 지금의 탑골공원 자리에 있었던 절로, 조선 세조 11년(1465)에 세웠다...",19621220.0,\r\n 국유\r\n,\r\n 종로구\r\n\t\t,조선시대 초기 15세기,1기,126.988208,37.571546,http://www.khs.go.kr/unisearch/images/national...,http://116.67.83.213/webdata/file_data/media_d...,11.0,20000000.0,11


In [42]:
df = dataset[['type', 'sido']]
df.columns

Index(['type', 'sido'], dtype='object')

In [43]:
print(df['type'].unique())
print(df['sido'].unique())

['국보' '보물' '사적' '명승' '천연기념물' '국가무형유산' '국가민속문화유산' '시도유형문화유산' '시도무형유산'
 '시도기념물' '시도민속문화유산' '시도등록문화유산' '문화유산자료' '시도자연유산' '국가등록문화유산' '자연유산자료'
 '궁궐·종묘']
['서울' '부산' '대구' '인천' '광주' '대전' '울산' '세종' '경기' '강원' '충북' '충남' '전북' '전남'
 '경북' '경남' '제주']


In [44]:
print(df['sido'].value_counts())  
print(df['type'].value_counts())

sido
경남    2368
경북    2291
서울    2235
전남    1389
경기    1309
충남    1155
전북    1034
충북     857
강원     656
부산     564
제주     402
대구     333
인천     288
대전     235
광주     168
울산     167
세종      64
Name: count, dtype: int64
type
시도유형문화유산    4224
문화유산자료      2913
보물          2411
시도기념물       1579
국가등록문화유산     972
시도무형유산       639
사적           563
시도민속문화유산     473
천연기념물        378
국보           360
국가민속문화유산     315
시도자연유산       193
명승           134
궁궐·종묘        124
국가무형유산       118
시도등록문화유산     106
자연유산자료        13
Name: count, dtype: int64


In [31]:
print(len(dataset))

15515


### 글자수로 쪼개기

In [6]:
import time
start = time.time()

from langchain_community.document_loaders import CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = CSVLoader('heritage_rag_full.csv', encoding='utf-8')
text_splitter = RecursiveCharacterTextSplitter( 
    chunk_size=1500, 
    chunk_overlap=200
)
document = loader.load_and_split(text_splitter=text_splitter)

runtime = time.time() - start

print('문서 쪼개면서 읽는 시간 :', runtime)

문서 쪼개면서 읽는 시간 : 1.8705334663391113


In [7]:
len(document[0].page_content)

1372

In [10]:
# chunk의 글자수
# print([len(doc.page_content) for doc in document])
print('chunk_max :',max([len(doc.page_content) for doc in document]), 'chunk_min :', min([len(doc.page_content) for doc in document]))

chunk_max : 1500 chunk_min : 99


# 쪼갠 문서를 임베딩 -> 벡터 데이터베이스 저장
- openai_text-embedding-3-large
- 벡터 데이터베이스 : chroma

In [11]:
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
load_dotenv()

embedding = OpenAIEmbeddings(
    model = 'text-embedding-3-large',
)

# openai 한번 요청 토큰수 최대 30만개라서 batch사이즈 조정해서 요청으로 변경

In [12]:
# %%time
# from langchain_chroma import Chroma
# # 데이터를 처음 저장할 때
# database = Chroma.from_documents(
#     documents=document,
#     embedding=embedding,
#     collection_name="tax_collection",
#     persist_directory='./chromadata'
# )

BadRequestError: Error code: 400 - {'error': {'message': 'Requested 811957 tokens, max 300000 tokens per request', 'type': 'max_tokens_per_request', 'param': None, 'code': 'max_tokens_per_request'}}

# 처음데이터를 저장할때

In [18]:
# from langchain_chroma import Chroma

# batch_size = 350  # 상황에 따라 100~1000 사이로 조절

# for i in range(0, len(document), batch_size):
#     batch = document[i:i+batch_size]
#     Chroma.from_documents(
#         documents=batch,
#         embedding=embedding,
#         collection_name="tax_collection",
#         persist_directory='./chromadata'
#     )

# 이미 저장된 vector DB를 사용할 때

In [19]:
database = Chroma(
    embedding_function=embedding,
    collection_name="tax_collection",
    persist_directory='./chromadata'
)

# langchain 사용 안할 경우

In [20]:
query = "제주도에서 유명한 문화재는 뭐가있나요?"
retrieved_docs = database.similarity_search(query,
                                           k=4) # 기본 k는 4

In [21]:
retrieved_docs[0]

Document(id='d511e2d7-e091-49b4-8318-6613bc4ff295', metadata={'source': './tax_docs/소득세법(법률)(제20615호)(20250701).docx'}, page_content='가. 「도시개발법」이나 그 밖의 법률에 따른 환지처분으로 지목 또는 지번이 변경되거나 보류지(保留地)로 충당되는 경우\n\n나. 토지의 경계를 변경하기 위하여 「공간정보의 구축 및 관리 등에 관한 법률」 제79조에 따른 토지의 분할 등 대통령령으로 정하는 방법과 절차로 하는 토지 교환의 경우\n\n다. 위탁자와 수탁자 간 신임관계에 기하여 위탁자의 자산에 신탁이 설정되고 그 신탁재산의 소유권이 수탁자에게 이전된 경우로서 위탁자가 신탁 설정을 해지하거나 신탁의 수익자를 변경할 수 있는 등 신탁재산을 실질적으로 지배하고 소유하는 것으로 볼 수 있는 경우\n\n2. “주식등”이란 주식 또는 출자지분을 말하며, 신주인수권과 대통령령으로 정하는 증권예탁증권을 포함한다.\n\n3. “주권상장법인”이란 「자본시장과 금융투자업에 관한 법률」 제9조제15항제3호에 따른 주권상장법인을 말한다.\n\n4. “주권비상장법인”이란 제3호에 따른 주권상장법인이 아닌 법인을 말한다.\n\n5. “실지거래가액”이란 자산의 양도 또는 취득 당시에 양도자와 양수자가 실제로 거래한 가액으로서 해당 자산의 양도 또는 취득과 대가관계에 있는 금전과 그 밖의 재산가액을 말한다.\n\n6. “1세대”란 거주자 및 그 배우자(법률상 이혼을 하였으나 생계를 같이 하는 등 사실상 이혼한 것으로 보기 어려운 관계에 있는 사람을 포함한다. 이하 이 호에서 같다)가 그들과 같은 주소 또는 거소에서 생계를 같이 하는 자[거주자 및 그 배우자의 직계존비속(그 배우자를 포함한다) 및 형제자매를 말하며, 취학, 질병의 요양, 근무상 또는 사업상의 형편으로 본래의 주소 또는 거소에서 일시 퇴거한 사람을 포함한다]와 함께 구성하는 가족단위를 말한다. 다만, 대통령령으로 정하는 경우에는 

## 유사도 검색으로 가져온 문서를 질문과 같이 LLM 전달하여 답변 생성
- langchain 사용 안할 경우

In [22]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4.1-nano")

In [23]:
prompt = f"""[idendity]
- 당신은 최고의 한국 문화재 전문가입니다.
- [context]를 참고해서 사용자의 질문에 답변해주세요.
[context]는 다음과 같습니다.
{retrieved_docs}
Question : {query}"""

In [24]:
ai_message = llm.invoke(prompt)

In [25]:
print(ai_message.content)

제주도는 한국의 대표적인 문화유산과 자연경관이 풍부한 섬으로, 다양한 유명한 문화재가 있습니다. 대표적인 문화재는 다음과 같습니다.

1. 성산 일출봉 (Seongsan Ilchulbong) : 유네스코 세계자연유산으로 등록된 화산 분화구로, 일출 풍경이 매우 아름다워 많은 관광객이 찾는 명소입니다.
2. 만장굴 (Manjanggul Cave) : 세계 최대 규모의 용암 동굴로, 천연 동굴 내에 형성된 다양한 석회화물과 기암괴석이 인상적입니다.
3. 제주 민속촌 (Jeju Folk Village) : 제주 전통 가옥과 생활 모습을 체험할 수 있는 민속문화공간으로, 제주 민속문화의 전통을 느낄 수 있습니다.
4. 지붕 없는 박물관, 돌하르방 (Stone Grandfathers) : 제주 곳곳에 세워진 돌하르방 조각상은 제주 민속의 상징이며, 문화재로도 가치가 높습니다.
5. 제주목관아 : 조선시대 제주 지역 행정을 담당하던 관아 건물로, 조선시대 역사와 행정 문화를 보여주는 중요한 문화재입니다.
6. 천지연 폭포와 주변 유적지 : 자연경관뿐만 아니라 여러 문화유적과 역사적 장소들이 함께 보존되어 있습니다.
7. 제주 향교 (Seogwipo Hyanggyo) : 조선시대 유교 교육기관으로, 전통 문화와 교육 역사를 느낄 수 있는 문화재입니다.

이외에도 다양한 돌과 유적지, 전통 마을 등 제주도는 자연과 역사, 문화를 아우르는 풍부한 문화재들을 간직하고 있습니다.


# Augmentation을 위한 제공되는 Prompt 활용하여 langchain으로 답변 생성

### RetrievalQA를 통해 LLM전달 (create_retrieval_chain이 대체)
```
  query -> retrievar전달(vector 검색 수행: 최적화된 유사도가 높은 데이터 산출) -> retrieval문서 -> {context}에 삽입
  -> query = prompt의 {question}에 삽입
```

In [27]:
query = "경주에 유명한 문화재는 뭐가있나요?"

from langchain import hub
prompt = hub.pull("rlm/rag-prompt")

# QA체인만들기
from langchain.chains import RetrievalQA
qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever = database.as_retriever(search_kwargs={'k':5}),
    chain_type_kwargs={"prompt":prompt}
)

In [28]:
ai_message = qa_chain.invoke({"query":query})

In [29]:
ai_message

{'query': '경주에 유명한 문화재는 뭐가있나요?',
 'result': '경주는 신라시대의 유물과 사적지가 많아 유명한 문화재가 많이 있습니다. 대표적으로 불국사와 석굴암이 있으며, 첨성대와 안압지 등도 유명합니다. 이 문화재들은 세계문화유산으로 등록되어 있어 많은 관광객이 방문합니다.'}

In [1]:
# CLIP imports 추가
from transformers import CLIPProcessor, CLIPModel
from langchain_upstage import UpstageEmbeddings
from dotenv import load_dotenv
from langchain_pinecone import PineconeVectorStore
import os
from PIL import Image
import torch
from huggingface_hub import login
import base64
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

load_dotenv()

login(token=os.getenv("HF_TOKEN"))
print("라이브러리 로딩 완료!")

# 2. 환경 설정 (기존과 동일)

embedding = UpstageEmbeddings(model="solar-embedding-1-large", api_key=os.getenv("UPSTAGE_API_KEY"))

# 3. CLIP 모델 초기화
print("CLIP 모델 로딩 중... (처음에는 시간이 걸릴 수 있습니다)")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
print("CLIP 모델 로딩 완료!")

# 4. Pinecone 설정
index_name = "upstage-index"
database = PineconeVectorStore(
    embedding=embedding,
    index_name=index_name,
)
retriever = database.as_retriever(search_kwargs={'k': 4})

# 5. 문화재 카테고리 매핑 (CLIP 분석용)
HERITAGE_CATEGORIES = {
    "Korean traditional palace building": {
        "korean": "궁궐",
        "keywords": ["궁궐", "궁전", "조선왕조", "경복궁", "창덕궁", "광화문", "대한문"],
        "description": "조선시대 왕궁 건축물"
    },
    "Korean Buddhist temple architecture": {
        "korean": "사찰",
        "keywords": ["사찰", "절", "불교", "대웅전", "법당", "산사", "템플"],
        "description": "불교 사원 건축물"
    },
    "Korean traditional house hanok": {
        "korean": "한옥",
        "keywords": ["한옥", "전통가옥", "기와집", "초가집", "민가", "전통건축"],
        "description": "한국 전통 주거 건축"
    },
    "Korean stone pagoda tower": {
        "korean": "석탑",
        "keywords": ["석탑", "탑", "다층탑", "불탑", "3층석탑", "5층석탑", "석조탑"],
        "description": "석조 불교 탑"
    },
    "Korean Buddha statue sculpture": {
        "korean": "불상",
        "keywords": ["불상", "부처님", "불교조각", "석불", "마애불", "금동불"],
        "description": "불교 조각상"
    },
    "Korean traditional pottery ceramics": {
        "korean": "도자기",
        "keywords": ["도자기", "청자", "백자", "분청사기", "고려청자", "조선백자"],
        "description": "전통 도자기"
    },
    "Korean traditional painting artwork": {
        "korean": "전통회화",
        "keywords": ["전통회화", "민화", "문인화", "산수화", "초상화", "불화"],
        "description": "한국 전통 그림"
    },
    "Korean fortress wall castle": {
        "korean": "성곽",
        "keywords": ["성곽", "성벽", "산성", "읍성", "행궁", "성문"],
        "description": "방어용 성곽 건축"
    }
}

# 6. CLIP 이미지 분석 함수
def analyze_image_with_clip(image_path: str) -> dict:
    """
    CLIP을 사용하여 이미지에서 문화재 유형을 분석
    
    Args:
        image_path: 분석할 이미지 파일 경로
    
    Returns:
        분석 결과 딕셔너리
    """
    try:
        print(f"이미지 분석 중: {image_path}")
        
        # 이미지 로드 및 검증
        if not os.path.exists(image_path):
            raise FileNotFoundError(f"이미지 파일을 찾을 수 없습니다: {image_path}")
        
        image = Image.open(image_path)
        if image.mode != 'RGB':
            image = image.convert('RGB')
        
        print(f"이미지 크기: {image.size}")
        
        # CLIP 분석용 카테고리 리스트
        categories = list(HERITAGE_CATEGORIES.keys())
        
        # CLIP 처리
        inputs = clip_processor(
            text=categories, 
            images=image, 
            return_tensors="pt", 
            padding=True
        )
        
        with torch.no_grad():
            outputs = clip_model(**inputs)
        
        # 유사도 계산
        logits_per_image = outputs.logits_per_image
        probs = logits_per_image.softmax(dim=1)
        
        # 상위 3개 결과 추출
        top3_indices = probs[0].topk(3).indices
        top3_probs = probs[0].topk(3).values
        
        results = []
        for i, (idx, prob) in enumerate(zip(top3_indices, top3_probs)):
            category_key = categories[idx.item()]
            category_info = HERITAGE_CATEGORIES[category_key]
            confidence = prob.item() * 100
            
            results.append({
                "rank": i + 1,
                "english_category": category_key,
                "korean_category": category_info["korean"],
                "description": category_info["description"],
                "confidence": confidence,
                "keywords": category_info["keywords"]
            })
        
        # 최고 예측 결과
        best_result = results[0]
        
        # print(f"분석 결과: {best_result['korean_category']} ({best_result['confidence']:.1f}%)")
        # print(f"상위 3개: {[r['korean_category'] for r in results]}")
        
        return {
            # "success": True,
            "best_prediction": best_result,
            # "top3_predictions": results,
            "search_keywords": best_result["keywords"][:10],  # 상위 4개 키워드
            # "search_query": " ".join(best_result["keywords"][:3])  # 검색용 쿼리
        }
        
    except Exception as e:
        print(f"이미지 분석 오류: {e}")
        return {
            "success": False,
            "error": str(e),
            "search_keywords": ["한국", "문화재", "유산"],
            "search_query": "한국 문화재"
        }

IMAGE_PATH="kbg.png"
with open(IMAGE_PATH, "rb") as f:
    img_b64 = base64.b64encode(f.read()).decode("utf-8")
# text = extract_image_chunks("sample5.png")
# 프롬프트와 이미지를 HumanMessage에 담기
img_data = analyze_image_with_clip(IMAGE_PATH).get("search_keywords",["문화", "유산"])
msg = HumanMessage(
    content=[
        # {"type": "text", "text": "이미지 속 표나 그래프를 마크다운 형태로 반환해주세요. 없는 내용은 추가하지 않으며 인식한 내용만 반영합니다."},
        {"type": "text", "text": "다음 이미지는 한국의 문화재입니다. 주어진 리스트는 이를 CLIP을 통해 예측한 결과입니다. 어떤 문화재 일까요?"},
        # {"type": "text", "text": f"다음은 OCR을 통해 추출한 내용을 정리한 것입니다. 참고자료로 활용해주세요.{text}"},
        # {"type": "text", "text": """다음 이미지는 작전 전투 지형에 관한 위성사진입니다.
        #  1. 해당 지형을 분석합니다. 어떤 형태의 지형인지, 적군이 잠복하기 좋은 위치는 어디인지 등에 대해 판별합니다.
        #  2. 해당 사진의 시가지, 개활지, 산지를 각각 몇 %인지 분석해주세요. 반환 형태는 아래와 같습니다.
        #  [반환형태]
        #  시가지 : xx%,
        #  개활지 : xx%,
        #  산지 : xx% """},
        {"type": "image_url", "image_url": f"data:image/png;base64,{img_b64}"},
        {"type": "text", "text": f"{img_data}"},
    ]
)
llm = ChatOllama(model="qwen2.5vl:7b")
# 결과 받기
response = llm.invoke([msg])
print(response.content)

if __name__ == "__main__" :
    print(analyze_image_with_clip('kbg.png'))

C:\Users\Admin\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Admin\anaconda3\envs\llm\lib\site-packages\torch\_subclasses\functional_tensor.py:295: UserWarning: Failed to initialize NumPy: DLL load failed while importing _multiarray_umath: 지정된 모듈을 찾을 수 없습니다. (Triggered internally at C:\cb\pytorch_1000000000000\work\torch\csrc\utils\tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


ImportError: DLL load failed while importing _multiarray_umath: 지정된 모듈을 찾을 수 없습니다.

ImportError: DLL load failed while importing _multiarray_umath: 지정된 모듈을 찾을 수 없습니다.

ImportError: DLL load failed while importing _multiarray_umath: 지정된 모듈을 찾을 수 없습니다.

ImportError: _multiarray_umath failed to import

ImportError: numpy._core.umath failed to import

In [2]:
import numpy as np
np.arange(0,10)

array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])